# Análisis exploratorio y recorrido del ETL
### PROYECTO ETL: Análisis de datos de apuestas deportivas

**Autores**

- Camilo Andrés Ardila Beleño — 834328
- Yesli Eliana Malaver Cangrejo — 854165
- Alan Gabo Sastre — 865413

> Proyecto académico con datos **históricos simulados**. No utiliza apuestas
> reales ni scraping de casas de apuestas. No promueve las apuestas reales.


## Objetivo del notebook

Recorrer de principio a fin el pipeline ETL y responder preguntas básicas:

1. ¿Cuántos registros y fuentes hay?
2. ¿Qué problemas de calidad se detectan y cómo se corrigen?
3. ¿Cómo queda el dataset limpio y el modelo estrella?
4. ¿Qué patrones muestran los eventos, los resultados y las cuotas?

**Requisito:** haber ejecutado `python main.py` al menos una vez. El notebook
puede regenerar los datos si no existen.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.config import FIGURES_DIR, PROCESSED_DIR, RAW_DIR, REPORTS_DIR
from src.extract import extraer
from src.generate_sample_data import generar_datos

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Raíz del proyecto:", RAIZ)


## 1. Datos de entrada

Los datos crudos se generan con `src/generate_sample_data.py` (semilla 42) e
incluyen duplicados, nulos, fechas mezcladas, typos y cuotas inválidas.


In [ ]:
if not any(RAW_DIR.glob("*.csv")) and not any(RAW_DIR.glob("*.xlsx")):
    generar_datos()

for ruta in sorted(RAW_DIR.iterdir()):
    if ruta.is_file() and not ruta.name.startswith("~$"):
        print(f"{ruta.name:28s} {ruta.stat().st_size / 1024:8.1f} KB")


In [ ]:
# Inspección de una fuente cruda: se ven formatos y errores mezclados
futbol_crudo = pd.read_csv(
    RAW_DIR / "eventos_futbol.csv", encoding="utf-8-sig", dtype=str
)
print("Filas de ejemplo en la fuente de fútbol:", len(futbol_crudo))
futbol_crudo.head(8)


## 2. Extract

`extraer()` descubre archivos, unifica columnas con alias, agrega
`fuente_archivo` y registra el conteo por fuente.


In [ ]:
df_crudo = extraer()
print("Registros extraídos:", len(df_crudo))
print("Columnas:", list(df_crudo.columns))
df_crudo.head()


In [ ]:
registros_por_fuente = (
    df_crudo.groupby("fuente_archivo").size().rename("registros").to_frame()
)
registros_por_fuente


## 3. Transform paso a paso

Se aplican las mismas funciones que ejecuta `main.py`, mostrando cómo cambia
la forma del dataset en cada etapa.


In [ ]:
from src.transform import (
    TransformStats,
    convertir_tipos,
    corregir_nombres_equipos,
    crear_columnas_derivadas,
    eliminar_duplicados,
    estandarizar_fechas,
    generar_reporte_calidad,
    organizar_columnas,
    tratar_nulos,
    validar_cuotas,
)

stats = TransformStats(
    filas_iniciales=len(df_crudo),
    nulos_antes=df_crudo.isna().sum().to_dict(),
)

df = eliminar_duplicados(df_crudo, stats)
print(f"Tras eliminar duplicados:      {df.shape}")

df = tratar_nulos(df, stats)
print(f"Tras tratar nulos:             {df.shape}")

df = estandarizar_fechas(df, stats)
print(f"Tras estandarizar fechas:      {df.shape}")

df = corregir_nombres_equipos(df, stats)
print(f"Equipos corregidos:            {stats.equipos_corregidos}")

df = validar_cuotas(df, stats)
print(f"Tras validar cuotas:           {df.shape}")

df = crear_columnas_derivadas(df, stats)
df = convertir_tipos(df, stats)
df = organizar_columnas(df)

stats.filas_finales = len(df)
stats.nulos_despues = df.isna().sum().to_dict()
print(f"Dataset final:                 {df.shape}")


In [ ]:
generar_reporte_calidad(df_crudo, df, stats, REPORTS_DIR)
display(Markdown((REPORTS_DIR / "reporte_calidad.md").read_text(encoding="utf-8")))


In [ ]:
resumen = pd.DataFrame(
    {
        "valor": [
            stats.filas_iniciales,
            stats.filas_finales,
            stats.duplicados_exactos,
            stats.duplicados_por_id,
            stats.fechas_invalidas,
            stats.equipos_corregidos,
            sum(stats.cuotas_invalidas.values()),
            sum(stats.cuotas_atipicas.values()),
            sum(stats.cuotas_imputadas.values()),
        ]
    },
    index=[
        "Filas crudas",
        "Filas limpias",
        "Duplicados exactos",
        "Duplicados por id",
        "Fechas inválidas",
        "Nombres corregidos",
        "Cuotas inválidas",
        "Cuotas atípicas",
        "Cuotas imputadas",
    ],
)
resumen


## 4. Load y modelo estrella

Se guardan el dataset limpio y las cinco tablas del modelo estrella, además de
la base SQLite opcional.


In [ ]:
from src.load import run_load

rutas = run_load(df, usar_sqlite=True)
pd.Series(
    {clave: str(ruta.relative_to(RAIZ)) for clave, ruta in rutas.items()},
    name="archivo",
).to_frame()


In [ ]:
import sqlite3

conexion = sqlite3.connect(PROCESSED_DIR / "apuestas.db")
consulta = '''
SELECT d.deporte,
       COUNT(*) AS eventos,
       ROUND(AVG(f.cuota_local), 2) AS cuota_local_promedio,
       ROUND(AVG(f.prob_implicita_local) * 100, 1) AS prob_local_pct
FROM fact_eventos f
JOIN dim_deportes d ON d.id_deporte = f.id_deporte
GROUP BY d.deporte
ORDER BY eventos DESC;
'''
pd.read_sql_query(consulta, conexion)


## 5. Análisis rápido del dataset limpio

Además de los gráficos, conviene revisar agregaciones directamente con Pandas.


In [ ]:
analisis_deporte = (
    df.groupby("deporte", observed=True)
    .agg(
        eventos=("id_evento", "size"),
        cuota_local_promedio=("cuota_local", "mean"),
        cuota_visitante_promedio=("cuota_visitante", "mean"),
        favorito_local_pct=("favorito", lambda s: (s == "Local").mean() * 100),
    )
    .round(2)
)
analisis_deporte


In [ ]:
resultados = (
    df["resultado_normalizado"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("porcentaje")
    .to_frame()
)
resultados


## 6. Visualizaciones generadas con Matplotlib

Las figuras están en `reports/figures/` y se muestran a continuación.


In [ ]:
for ruta in sorted(FIGURES_DIR.glob("*.png")):
    display(Markdown(f"**{ruta.stem.replace('_', ' ').title()}**"))
    display(Image(filename=str(ruta)))


## 7. Conclusiones

- El dataset crudo tenía **duplicados exactos y por identificador**, nulos,
  fechas imposibles, nombres inconsistentes y cuotas inválidas.
- Después del ETL se conserva más del 95 % de los registros y se eliminan
  únicamente aquellos que no son trazables (sin id o sin fecha válida).
- El modelo estrella permite analizar los eventos por deporte, competición,
  equipo y fecha sin ambigüedades.
- Las cuotas presentan una distribución asimétrica: la mediana es menor que la
  media y existen valores extremos que fueron acotados.
- El dashboard de Power BI puede construirse siguiendo
  `powerbi/GUIA_DASHBOARD.md` y las medidas de `powerbi/medidas_DAX.md`.

> Recordatorio: los datos son simulados y el proyecto tiene fines educativos.
> No promueve las apuestas reales.
